Imports

In [4]:
import pandas as pd
import numpy as np
import json
import joblib

from sentence_transformers import SentenceTransformer, util

Load fine-tuned MPNet model

In [5]:
mpnet_model_path = r"D:\MY PROJECTS\CS PROJECT 2\Red_Hope\CONVERSATIONAL MODELS\mpnet_donor_finetuned"

retriever_model = SentenceTransformer(mpnet_model_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Load QA dataset

In [6]:
qa_df = pd.read_csv("blood_donation_qa_final.csv")

qa_df.head()

qa_df.columns

Index(['Question', 'Answer'], dtype='str')

Prepare corpus

In [7]:
questions = qa_df["Question"].astype(str).tolist()
answers = qa_df["Answer"].astype(str).tolist()

Create embeddings

In [8]:
question_embeddings = retriever_model.encode(
    questions,
    convert_to_tensor=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

MPNet retrieval function

In [9]:
def mpnet_retriever(user_query, top_k=3):
    query_embedding = retriever_model.encode(
        user_query,
        convert_to_tensor=True
    )

    scores = util.cos_sim(query_embedding, question_embeddings)[0]
    top_results = scores.topk(k=top_k)

    results = []

    for score, idx in zip(top_results.values, top_results.indices):
        idx = int(idx)
        results.append({
            "question": questions[idx],
            "answer": answers[idx],
            "similarity": float(score)
        })

    return results

Confidence formatter

In [10]:
def assign_confidence(similarity):
    if similarity >= 0.93:
        return "Very High"
    elif similarity >= 0.88:
        return "High"
    elif similarity >= 0.78:
        return "Moderate"
    else:
        return "Low"

General QA response

In [11]:
def answer_general_question(user_query):
    results = mpnet_retriever(user_query, top_k=3)

    best = results[0]
    confidence = assign_confidence(best["similarity"])

    if confidence in ["Very High", "High"]:
        return {
            "mode": "general_qa",
            "answer": best["answer"],
            "matched_question": best["question"],
            "similarity": round(best["similarity"], 4),
            "confidence": confidence
        }

    elif confidence == "Moderate":
        return {
            "mode": "general_qa",
            "answer": best["answer"],
            "matched_question": best["question"],
            "similarity": round(best["similarity"], 4),
            "confidence": confidence,
            "note": "This answer may need confirmation from a health worker."
        }

    else:
        return {
            "mode": "general_qa",
            "answer": "I am not confident enough to answer this accurately. Please consult a blood donation health worker.",
            "matched_question": best["question"],
            "similarity": round(best["similarity"], 4),
            "confidence": confidence
        }

Connect MPNet with unified router

In [14]:
def detect_intent(user_query):
    query = user_query.lower()

    eligibility_keywords = [
        "eligible",
        "can i donate",
        "allowed to donate",
        "hemoglobin",
        "weight",
        "medical condition",
        "sick",
        "illness",
        "condition"
    ]

    availability_keywords = [
        "available",
        "likely to donate",
        "donate again",
        "return",
        "campaign",
        "invite",
        "reminder"
    ]

    if any(word in query for word in eligibility_keywords) and any(word in query for word in availability_keywords):
        return "combined"

    if any(word in query for word in eligibility_keywords):
        return "eligibility"

    if any(word in query for word in availability_keywords):
        return "availability"

    return "general_qa"

In [18]:
import json
import joblib
import pandas as pd

# Load eligibility rules
with open("eligibility_rules.json", "r") as file:
    eligibility_rules = json.load(file)

# Load availability model
availability_model = joblib.load("availability_logistic_model.pkl")


def eligibility_engine(donor, rules):
    condition = donor.get("medical_condition", "None")
    weight = donor.get("weight_kg")
    hemoglobin = donor.get("hemoglobin_g_dl")

    if condition in rules["disallowed_conditions"]:
        return {
            "eligible": False,
            "reason": f"Medical condition '{condition}' prevents donation.",
            "decision": "Not Eligible"
        }

    if weight is not None and weight < rules["min_weight_kg"]:
        return {
            "eligible": False,
            "reason": f"Weight below minimum requirement ({rules['min_weight_kg']} kg).",
            "decision": "Not Eligible"
        }

    if hemoglobin is not None and hemoglobin < rules["min_hemoglobin_g_dl"]:
        return {
            "eligible": False,
            "reason": f"Hemoglobin below minimum requirement ({rules['min_hemoglobin_g_dl']} g/dL).",
            "decision": "Not Eligible"
        }

    return {
        "eligible": True,
        "reason": "Donor satisfies eligibility requirements.",
        "decision": "Eligible"
    }


def availability_engine(donor, availability_model):
    features = pd.DataFrame([{
        "Recency": donor.get("recency"),
        "Frequency": donor.get("frequency"),
        "Time": donor.get("time")
    }])

    probability = availability_model.predict_proba(features)[0][1]
    prediction = availability_model.predict(features)[0]

    return {
        "available": bool(prediction == 1),
        "probability": round(probability, 4),
        "decision": "Likely Available" if prediction == 1 else "Not Likely Available"
    }


def combined_decision(eligibility_result, availability_result):
    eligible = eligibility_result["eligible"]
    available = availability_result["available"]

    if eligible and available:
        action = "Invite donor for the next campaign."
    elif eligible and not available:
        action = "Send soft reminder or awareness message."
    elif not eligible and available:
        action = "Do not invite yet. Provide medical guidance first."
    else:
        action = "Low priority for campaign invitation."

    return {
        "eligibility": eligibility_result["decision"],
        "availability": availability_result["decision"],
        "availability_probability": availability_result["probability"],
        "reason": eligibility_result["reason"],
        "recommended_action": action
    }


def format_response(result):
    return f"""
Eligibility Status: {result['eligibility']}
Availability Status: {result['availability']}
Availability Probability: {result['availability_probability']}

Reason:
{result['reason']}

Recommended Action:
{result['recommended_action']}
"""

In [19]:
def unified_chatbot_response(user_query, donor=None):
    intent = detect_intent(user_query)

    if intent == "general_qa":
        return answer_general_question(user_query)

    if intent == "eligibility":
        if donor is None:
            return {
                "mode": "eligibility",
                "answer": "Please provide donor medical condition, weight, and hemoglobin level."
            }

        eligibility_result = eligibility_engine(donor, eligibility_rules)

        return {
            "mode": "eligibility",
            "answer": eligibility_result["reason"],
            "decision": eligibility_result["decision"]
        }

    if intent == "availability":
        if donor is None:
            return {
                "mode": "availability",
                "answer": "Please provide recency, frequency, and time values for availability prediction."
            }

        availability_result = availability_engine(donor, availability_model)

        return {
            "mode": "availability",
            "answer": f"{availability_result['decision']} with probability {availability_result['probability']}.",
            "decision": availability_result["decision"],
            "probability": availability_result["probability"]
        }

    if intent == "combined":
        if donor is None:
            return {
                "mode": "combined",
                "answer": "Please provide eligibility and availability details for this donor."
            }

        eligibility_result = eligibility_engine(donor, eligibility_rules)
        availability_result = availability_engine(donor, availability_model)
        final_result = combined_decision(eligibility_result, availability_result)

        return {
            "mode": "combined",
            "answer": format_response(final_result),
            "result": final_result
        }

Test general QA

In [20]:
unified_chatbot_response("What are the benefits of donating blood?")

{'mode': 'general_qa',
 'answer': 'Blood banks can promote the benefits by using educational campaigns, social media, and community outreach to raise awareness. Clear communication about the impact of donations can encourage participation.',
 'matched_question': 'How can blood banks promote the benefits of automated blood collection to the public?',
 'similarity': 0.9356,
 'confidence': 'Very High'}

Test eligibility

In [21]:
sample_donor = {
    "medical_condition": "None",
    "weight_kg": 62,
    "hemoglobin_g_dl": 13.1,
    "recency": 2,
    "frequency": 8,
    "time": 24
}

unified_chatbot_response(
    "Am I eligible to donate blood?",
    donor=sample_donor
)

{'mode': 'eligibility',
 'answer': 'Donor satisfies eligibility requirements.',
 'decision': 'Eligible'}

Test combined reasoning

In [22]:
unified_chatbot_response(
    "Should we invite this donor for the next campaign?",
    donor=sample_donor
)

{'mode': 'availability',
 'answer': 'Likely Available with probability 0.734.',
 'decision': 'Likely Available',
 'probability': np.float64(0.734)}